# Data Quality Checks

In [24]:
import pandas as pd

df = pd.read_csv("../data/raw/upi_transactions_raw.csv")

df.head()

,transaction_id,user_id,timestamp,amount,transaction_type,merchant_category,merchant_id,city,device_id,status,receiver_id
0,TXN00037414,U01112,01-01-2026 00:08,289.59,P2M,Healthcare,M01383,Delhi,D00790,SUCCESS,M01383
1,TXN00029422,U01252,01-01-2026 00:11,77.56,P2M,Fuel,M00389,Ahmedabad,D01414,SUCCESS,M00389
2,TXN00026757,U00488,01-01-2026 00:11,154.65,P2P,P2P,NaN,Mumbai,D04419,SUCCESS,U01599
3,TXN00005630,U02139,01-01-2026 00:22,273.63,P2P,P2P,NaN,Ahmedabad,D03775,SUCCESS,U03408
4,TXN00009067,U02122,01-01-2026 00:27,93.49,P2P,P2P,NaN,Chennai,D03238,SUCCESS,U04199


In [48]:
print("--- Missing values in the dataset ---")
print(df.isnull().sum())

--- Missing values in the dataset ---
transaction_id           0
user_id                  0
timestamp                0
amount                   0
transaction_type         0
merchant_category        0
merchant_id          17692
city                     0
device_id                0
status                   0
receiver_id              0
dtype: int64


In [49]:
print("--- Duplicate transactions in the dataset ---")
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate transaction IDs:", df["transaction_id"].duplicated().sum())

--- Duplicate transactions in the dataset ---
Duplicate rows: 0
Duplicate transaction IDs: 0


In [56]:
print("--- Data types of the dataset ---")
print(df.dtypes)

--- Data types of the dataset ---
transaction_id                  str
user_id                         str
timestamp            datetime64[us]
amount                      float64
transaction_type                str
merchant_category               str
merchant_id                     str
city                            str
device_id                       str
status                          str
receiver_id                     str
dtype: object


In [62]:
print("Zero amounts:", (df["amount"] == 0).sum())
print("Negative amounts:", (df["amount"] < 0).sum())

Zero amounts: 0
Negative amounts: 0


In [72]:
print("--- Overview of transaction types ---")
df["transaction_type"].value_counts()
df.groupby("transaction_type")["amount"].agg(
    ["count", "mean", "median", "sum"]
)

--- Overview of transaction types ---


,count,mean,median,sum
transaction_type,,,,
P2M,32308,289.399890,201.22,9349931.63
P2P,17692,285.892664,199.97,5058013.02


In [75]:
print("--- User Activity Summary ---")
user_activity = (
    df.groupby("user_id")
      .agg(
          transaction_count=("transaction_id", "count"),
          total_amount=("amount", "sum"),
          average_amount=("amount", "mean"),
          max_amount=("amount", "max")
      )
      .sort_values(
          "transaction_count",
          ascending=False
      )
)

user_activity.head(20)

--- User Activity Summary ---


,transaction_count,total_amount,average_amount,max_amount
user_id,,,,
U04461,27,6345.38,235.014074,779.90
U04702,23,6644.35,288.884783,1160.08
U03397,22,6950.79,315.945000,1366.07
U02095,22,6336.07,288.003182,865.84
U00428,21,9589.09,456.623333,1855.52
U04163,21,5514.48,262.594286,716.62
U04303,21,4809.35,229.016667,837.53
U00245,21,6886.91,327.948095,1381.38
U01417,21,3851.48,183.403810,563.39


In [77]:
print("--- Data Cleaning Process ---")
# Make a copy so the raw data stays unchanged
clean_df = df.copy()

# Convert timestamp into a proper datetime column
clean_df["timestamp"] = pd.to_datetime(
    clean_df["timestamp"],
    errors="coerce"
)

# Make sure amount is numeric
clean_df["amount"] = pd.to_numeric(
    clean_df["amount"],
    errors="coerce"
)

# Remove rows where the essential transaction information is invalid
clean_df = clean_df.dropna(
    subset=[
        "transaction_id",
        "user_id",
        "timestamp",
        "amount",
        "transaction_type",
        "status"
    ]
)

# Remove duplicate transaction IDs
clean_df = clean_df.drop_duplicates(
    subset="transaction_id"
)

# Sort transactions by time
clean_df = clean_df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Raw rows:", len(df))
print("Clean rows:", len(clean_df))
print("Rows removed:", len(df) - len(clean_df))

--- Data Cleaning Process ---
Raw rows: 50000
Clean rows: 50000
Rows removed: 0


In [79]:
print("--- Save the cleaned dataset ---")
clean_df.to_csv(
    "../data/clean/upi_transactions_clean.csv",
    index=False
)
print("Clean dataset saved successfully.")

--- Save the cleaned dataset ---
Clean dataset saved successfully.
